# 1. Setup & Environment
- 대용량 공간 OD 행렬 및 중력 모델 고속 연산용 라이브러리 로드
- 부동소수점 포맷 및 판다스 디스플레이 환경 설정

In [1]:
# 1. 라이브러리 로드 및 환경 설정
from pathlib import Path
import json
import math
import re
import unicodedata
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
pd.set_option("display.float_format", lambda x: "%.4f" % x)
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 20)

# 2. Configuration & Parameter Definitions
- 입출력 디렉터리 경로(OD Matrix, 아파트 플래그, 충전소 스냅샷) 설정
- 랩미팅 확정 파라미터 정의: 2021~2024년 평일 낮(week_낮_normal) 단일 시나리오, 15분(900초) 임계치

In [2]:
# 2. 경로 및 분석 파라미터 정의
BASE_DIR = Path("/mnt/cowork/EV")
GAUSSIAN_OD_DIR = BASE_DIR / "output/g2sfca_sfast_gaussian"
CHARGER_DIR_FASTONLY = BASE_DIR / "input/processed/yearly_snapshots_fastonly"
APT_FP = BASE_DIR / "output/apt_charger_flags/seoul_chargers_2024_apt_v3_final.csv"

DIR_OUTPUT = BASE_DIR / "output/gravity_model_gaussian"
DIR_OUTPUT.mkdir(parents=True, exist_ok=True)

# 모델링 파라미터 (2026-08-27 랩미팅 확정 단일 시나리오)
CUTOFF_SEC = 900  # 15분 (900초)
YEARS = [2021, 2022, 2023, 2024]
DAYTYPE, PERIOD, SCENARIO = "week", "낮", "normal"
WINDOW_START, WINDOW_END = 11, 13

print(f">> 분석 모델: Gravity Model (Gaussian Decay, t0={CUTOFF_SEC}s)")
print(f">> 분석 시나리오: {YEARS}개년 | {DAYTYPE}_{PERIOD}_{SCENARIO} (운영시간 윈도우: {WINDOW_START}~{WINDOW_END}시)")
print(f">> 출력 디렉터리: {DIR_OUTPUT}")

>> 분석 모델: Gravity Model (Gaussian Decay, t0=900s)
>> 분석 시나리오: [2021, 2022, 2023, 2024]개년 | week_낮_normal (운영시간 윈도우: 11~13시)
>> 출력 디렉터리: /mnt/cowork/EV/output/gravity_model_gaussian


# 3. Distance Decay & Operating Hours Functions
- 충전소 운영시간 텍스트 정규식 파서 및 개방 여부 판별 함수
- 가우시안 거리 감쇄 함수(Gaussian Distance Decay) 벡터화 정의

In [3]:
# 3. 보조 연산 및 감쇄 함수 정의
TIME_RANGE_RE = re.compile(r"(\d{1,2})[:시](\d{2})?\s*[~-]\s*(\d{1,2})[:시](\d{2})?")

def parse_open_window(text):
    """운영시간 텍스트에서 시작/종료 시각 및 주중 전용 여부 추출"""
    if not text or not str(text).strip() or "24시간" in str(text) or "24시" in str(text):
        return None
    t = str(text).strip()
    m = TIME_RANGE_RE.search(t)
    if not m:
        return None
    h1, _, h2, _ = m.groups()
    start, end = int(h1), int(h2)
    weekday_only = ("평일" in t) or ("주중" in t)
    return (start, end, weekday_only)


def is_open(parsed, window_start, window_end, daytype="week"):
    """분석 대상 시간대 윈도우 내 충전소 개방 여부 판정"""
    if parsed is None:
        return True
    start, end, weekday_only = parsed
    if weekday_only and daytype == "weekend":
        return False
    if end <= start:
        return True
    return not (end <= window_start or start >= window_end)


def decay_gaussian(tt, d0=CUTOFF_SEC):
    """벡터화된 가우시안 연속 거리 감쇄 함수"""
    tt = np.asarray(tt, dtype=np.float64)
    w = (np.exp(-0.5 * (tt / d0)**2) - math.exp(-0.5)) / (1.0 - math.exp(-0.5))
    return np.where(tt <= d0, w, 0.0)

# 4. Vectorized Gravity Model Engine
- 수요(경쟁) 정규화(Step 1, R_j) 없이 유효 공급량(S_j)과 거리 감쇄를 집계구별로 직접 가중합산
- 수식: $A_i = \sum_{j} S_{j, \text{eff}} \cdot f(t_{ij})$

In [4]:
# 4. 고속 벡터화 Gravity Model 접근성 연산 엔진
def compute_gravity_accessibility(df_od: pd.DataFrame, dict_effective_supply: dict, d0: int = CUTOFF_SEC) -> pd.DataFrame:
    """OD 행렬과 유효 공급량 딕셔너리를 입력받아 집계구별 Gravity 접근성 점수 산출"""
    # 1. 가우시안 감쇄 가중치 벡터 연산
    w = decay_gaussian(df_od["travel_time_sec"].values, d0=d0)
    
    # 2. 유효 공급량 매핑 및 도달 공급 가중치 산출
    s_vals = df_od["station_id"].map(dict_effective_supply).fillna(0.0).values
    w_supply = s_vals * w
    
    # 3. 집계구별 단순 가중합산 (인구 수요 경쟁 정규화 제외)
    df_temp = pd.DataFrame({
        "oa_code": df_od["oa_code"].values,
        "w_supply": w_supply
    })
    
    df_acc = df_temp.groupby("oa_code", as_index=False)["w_supply"].sum()
    df_acc.rename(columns={"w_supply": "accessibility_score"}, inplace=True)
    return df_acc

# 5. Data Loaders
- 아파트 충전소(v3) 제외 식별자 로더
- 연도별 급속 충전기 GeoJSON 공급량 및 운영시간 로더

In [5]:
# 5. 충전소 기초 데이터 로더
df_apt = pd.read_csv(APT_FP, dtype={"station_id": str})
apt_set = set(df_apt[df_apt["is_apt_v3"]]["station_id"])
print(f">> 아파트 완전 제외 대상 충전소: {len(apt_set):,}개")

def load_supply_and_hours(year: int):
    """연도별 서울시 급속충전기 스냅샷에서 공급 용량 및 운영시간 로드"""
    fname = f"metro7_ev_chargers_{year}_fastonly.geojson"
    fp = unicodedata.normalize("NFD", str(CHARGER_DIR_FASTONLY / fname))
    with open(fp, encoding="utf-8") as f:
        data = json.load(f)
    supply, hours = {}, {}
    for feat in data["features"]:
        p = feat["properties"]
        if p.get("city") == "서울특별시":
            sid = str(p["station_id"])
            supply[sid] = float(p.get("fast_count", 0) or 0)
            hours[sid] = parse_open_window(p.get("openinghour", ""))
    return supply, hours

>> 아파트 완전 제외 대상 충전소: 5,634개


# 6. Batch Gravity Model Execution Pipeline
- 2021~2024년 4개년 대상 평일 낮(week_낮_normal) 중력 모델 일괄 산출
- 유효 공급량(아파트 제외 + 실측 운영시간 필터) 반영 및 결과 파일(`_mw.csv`) 저장

In [6]:
# 6. 4개년 배치 실행 및 CSV 내보내기
print("=" * 85)
print("RUNNING: GRAVITY MODEL (COMPETITION-FREE) ACCESSIBILITY PIPELINE")
print("=" * 85)

summary_records = []

for year in YEARS:
    raw_supply, hours_dict = load_supply_and_hours(year)
    tag = f"{year}_{DAYTYPE}_{PERIOD}_{SCENARIO}"
    fp_od = GAUSSIAN_OD_DIR / f"od_{tag}.csv"
    
    if not fp_od.exists():
        print(f"  [!] OD Matrix 파일 누락: {fp_od.name}")
        continue
        
    df_od = pd.read_csv(fp_od, dtype={"station_id": str, "oa_code": str})
    
    # 유효 공급량 산출 (아파트 제외 + 분석 윈도우 미운영 제외)
    dict_effective_supply = {}
    n_excluded_hours = 0
    for sid, count in raw_supply.items():
        if sid in apt_set:
            dict_effective_supply[sid] = 0.0
        elif not is_open(hours_dict.get(sid), WINDOW_START, WINDOW_END, DAYTYPE):
            dict_effective_supply[sid] = 0.0
            n_excluded_hours += 1
        else:
            dict_effective_supply[sid] = count
            
    # 고속 벡터화 Gravity 점수 산출
    df_result = compute_gravity_accessibility(df_od, dict_effective_supply, d0=CUTOFF_SEC)
    
    # 신규 표준 산출물 저장 (_mw.csv)
    out_fp_mw = DIR_OUTPUT / f"gravity_score_{tag}_mw.csv"
    df_result.to_csv(out_fp_mw, index=False, encoding="utf-8-sig")
    
    mean_score = df_result["accessibility_score"].mean()
    std_score = df_result["accessibility_score"].std()
    max_score = df_result["accessibility_score"].max()
    
    print(f"  [>] {tag} | 집계구: {len(df_result):,d}개 | 평균 점수: {mean_score:8.4f} | (운영시간 제외: {n_excluded_hours:3d}개소) -> {out_fp_mw.name}")
    
    summary_records.append({
        "year": year,
        "scenario": tag,
        "n_oa": len(df_result),
        "mean_score": mean_score,
        "std_score": std_score,
        "max_score": max_score,
        "n_excluded_hours": n_excluded_hours
    })

df_summary = pd.DataFrame(summary_records)

RUNNING: GRAVITY MODEL (COMPETITION-FREE) ACCESSIBILITY PIPELINE
  [>] 2021_week_낮_normal | 집계구: 19,153개 | 평균 점수:  26.3055 | (운영시간 제외:   2개소) -> gravity_score_2021_week_낮_normal_mw.csv
  [>] 2022_week_낮_normal | 집계구: 19,153개 | 평균 점수:  46.5665 | (운영시간 제외:   4개소) -> gravity_score_2022_week_낮_normal_mw.csv
  [>] 2023_week_낮_normal | 집계구: 19,153개 | 평균 점수:  68.7671 | (운영시간 제외:   4개소) -> gravity_score_2023_week_낮_normal_mw.csv
  [>] 2024_week_낮_normal | 집계구: 19,153개 | 평균 점수: 106.0030 | (운영시간 제외:   4개소) -> gravity_score_2024_week_낮_normal_mw.csv


# 7. Summary Pivot & Longitudinal Trends
- 2021~2024년 평일 낮 중력 모델 접근성 시계열 추이 요약표 출력

In [7]:
# 7. 기술통계 및 연도별 변화 추이 테이블 출력
print("\n" + "=" * 80)
print("             연도별 Gravity Model 접근성 시계열 추이 요약표 (2021~2024)")
print("=" * 80)
display(df_summary[["year", "scenario", "n_oa", "mean_score", "std_score", "max_score", "n_excluded_hours"]])


             연도별 Gravity Model 접근성 시계열 추이 요약표 (2021~2024)


,year,scenario,n_oa,mean_score,std_score,max_score,n_excluded_hours
0,2021,2021_week_낮_normal,19153,26.3055,14.3238,98.9316,2
1,2022,2022_week_낮_normal,19153,46.5665,20.3866,126.9242,4
2,2023,2023_week_낮_normal,19153,68.7671,33.1494,237.5075,4
3,2024,2024_week_낮_normal,19153,106.0030,46.8971,315.5466,4


# 8. Result Validation (Comparison with Original Outputs)
- 기존 원본 중력 모델 산출물(`gravity_score_{tag}.csv`)과 신규 산출물(`_mw.csv`) 간 수치 오차 전수 대조

In [8]:
# 8. 원본 산출물 vs 신규 산출물(_mw) 정밀 오차 검증
validation_records = []

for year in YEARS:
    tag = f"{year}_{DAYTYPE}_{PERIOD}_{SCENARIO}"
    fp_orig = DIR_OUTPUT / f"gravity_score_{tag}.csv"
    fp_new = DIR_OUTPUT / f"gravity_score_{tag}_mw.csv"
    
    if not fp_new.exists():
        continue
        
    if not fp_orig.exists():
        validation_records.append({
            "Scenario": tag,
            "Status": "원본 파일 없음 (비교 스킵)",
            "Max Abs Diff": np.nan,
            "Mean Abs Diff": np.nan
        })
        continue
        
    df_orig = pd.read_csv(fp_orig, dtype={"oa_code": str})
    df_new = pd.read_csv(fp_new, dtype={"oa_code": str})
    
    comp = df_orig.merge(df_new, on="oa_code", suffixes=("_orig", "_new"))
    diff = (comp["accessibility_score_orig"] - comp["accessibility_score_new"]).abs()
    
    max_diff = diff.max()
    mean_diff = diff.mean()
    
    validation_records.append({
        "Scenario": tag,
        "Status": "완전 일치 (정상)" if max_diff < 1e-5 else "오차 발생 확인 필요",
        "Max Abs Diff": max_diff,
        "Mean Abs Diff": mean_diff
    })

df_val = pd.DataFrame(validation_records)
print("=" * 80)
print("           Gravity Model 원본 vs 리팩토링 코드 수치 검증 요약표")
print("=" * 80)
display(df_val)

if df_val["Max Abs Diff"].dropna().max() < 1e-5:
    print(">> [판정] 모든 연도의 Gravity Model 점수가 기존 원본 산출물과 수학적으로 완벽히 일치합니다.")

           Gravity Model 원본 vs 리팩토링 코드 수치 검증 요약표


,Scenario,Status,Max Abs Diff,Mean Abs Diff
0,2021_week_낮_normal,완전 일치 (정상),0.0000,0.0000
1,2022_week_낮_normal,완전 일치 (정상),0.0000,0.0000
2,2023_week_낮_normal,완전 일치 (정상),0.0000,0.0000
3,2024_week_낮_normal,완전 일치 (정상),0.0000,0.0000


>> [판정] 모든 연도의 Gravity Model 점수가 기존 원본 산출물과 수학적으로 완벽히 일치합니다.
